## Step 2: The Bouncer at the TESS Data Club (ID Validation Pipeline)

Not every TIC ID in our list will actually have usable TESS data behind it — some stars weren't observed, some searches time out, some light curves are garbage. This notebook is the "quality control" gatekeeper: it goes through every ID, tries to download and clean its light curve, and sorts IDs into "this one works" vs. "this one doesn't" piles *before* we waste time doing anything more expensive with them.

**Why this matters:** Downloading and preprocessing light curve data from MAST (via `lightkurve`) can be slow and occasionally flaky (network issues, missing sectors, corrupted files). Rather than discovering failures mid-training in a later notebook, this step front-loads the pain and produces a clean, pre-validated list to build on.

**The Algorithm:**

1. **`download_lightcurve(tess_id)`**: searches for a TESS light curve via `lightkurve.search_lightcurve(f"TIC {tess_id}", mission="TESS")`. If the search returns zero results, it raises an exception immediately rather than silently returning `None` — failing loud and early so the caller's `try/except` can catch it and log the failure properly.

2. **`preprocess_lightcurve(lc)`**: runs the downloaded light curve through a short but meaningful cleaning chain:
   - `remove_nans()` — strips missing flux values.
   - `normalize()` — puts flux on a relative scale (median = 1) so brightness units become comparable across different stars/sectors.
   - `flatten(window_length=401)` — a `lightkurve`-native detrending step, smoothing out long-term stellar variability using a Savitzky-Golay-style filter under the hood, conceptually the same idea as the manual `savgol_filter` calls in the Kepler pipeline, just using `lightkurve`'s built-in convenience method instead of hand-rolling it.
   - `remove_outliers(sigma=5)` — clips extreme flux spikes (cosmic rays, instrumental artifacts) that sit more than 5 standard deviations from the trend.

3. **`process_star(tess_id)`**: the tidy combo of the above two — download, then clean. If *either* step throws, the exception bubbles up to the caller.

4. **Persistence helpers**:
   - `load_existing_data()` reads any previously saved `successful_tess_ids.csv` / `failed_tess_ids.csv`, reconstructing which **original row numbers** have already been processed — this is the resume/checkpoint mechanism that lets you stop and restart a long-running validation job without starting from scratch.
   - `save_ids()` writes the current successful/failed lists back out to those same two CSVs, each including an `original_row` column so you always know exactly where in the source file a given ID came from.

5. **Interactive control flow helpers** — a small trio of input-validation loops (`get_user_choice`, `get_start_row`, `get_how_many`, `continue_processing`) that handle:
   - Choosing between **starting fresh** (deletes old success/fail CSVs) or **appending** (resumes from where you left off).
   - Picking an arbitrary **starting row** to resume from, if you want to skip ahead manually.
   - Choosing **batch sizes** so you don't have to commit to processing all 500 IDs in one uninterruptible sitting.
   - A "process more?" prompt between batches — giving you an easy stopping point after each chunk.

6. **`main()`** ties it all together:
   - Loads the master list from `TESS_IDs.csv`.
   - Figures out which rows are genuinely unprocessed (skipping anything already marked successful or failed).
   - Processes IDs in **user-defined batches**, calling `process_star()` on each, sorting the result into the successful or failed bucket based on whether an exception was raised.
   - **Saves progress after every batch** (not just at the end) — so a crash, network drop, or Jupyter kernel restart mid-run doesn't cost you all your progress, just the current batch.
   - Prints a running tally (processed/successful/failed/remaining) after each batch, plus a final validation summary with a success-rate percentage and a preview of the first 10 successful and failed IDs.

**Output:** `successful_tess_ids.csv` and `failed_tess_ids.csv`, each with `original_row` + `tess_id` columns — the successful list becomes the validated input for the labeling stage (`02_dataset_2.ipynb`), while the failed list is a handy "don't bother re-trying these" blocklist.

*TL;DR: this notebook is the ID checkpoint at security — some stars get through, some get pulled aside for "additional screening" (i.e. thrown in `failed_tess_ids.csv`).*

In [ ]:
import numpy as np
import pandas as pd
import lightkurve as lk
import os
import warnings

warnings.filterwarnings('ignore')

In [ ]:
def download_lightcurve(tess_id):
    search = lk.search_lightcurve(f"TIC {tess_id}", mission="TESS")
    
    if len(search) == 0:
        raise Exception(f"No lightcurve found for TIC {tess_id}")
    
    lc = search.download()
    return lc

In [ ]:
def process_star(tess_id):
    lc = download_lightcurve(tess_id)
    clean_lc = preprocess_lightcurve(lc)
    return clean_lc

In [ ]:
def preprocess_lightcurve(lc):
    lc = lc.remove_nans()
    lc = lc.normalize()
    lc = lc.flatten(window_length=401)
    lc = lc.remove_outliers(sigma=5)
    return lc

In [ ]:
def save_ids(successful_rows, successful_ids, failed_rows, failed_ids):
    if successful_ids:
        df_success = pd.DataFrame({
            'original_row': successful_rows,
            'tess_id': successful_ids
        })
        df_success.to_csv('successful_tess_ids.csv', index=False)
    
    if failed_ids:
        df_failed = pd.DataFrame({
            'original_row': failed_rows,
            'tess_id': failed_ids
        })
        df_failed.to_csv('failed_tess_ids.csv', index=False)

In [ ]:
def load_existing_data():
    successful_ids = []
    failed_ids = []
    successful_rows = []
    failed_rows = []
    processed_rows = set()
    
    if os.path.exists('successful_tess_ids.csv'):
        df = pd.read_csv('successful_tess_ids.csv')
        if 'tess_id' in df.columns:
            successful_ids = df['tess_id'].tolist()
        if 'original_row' in df.columns:
            successful_rows = df['original_row'].tolist()
            processed_rows.update(df['original_row'].tolist())
    
    if os.path.exists('failed_tess_ids.csv'):
        df = pd.read_csv('failed_tess_ids.csv')
        if 'tess_id' in df.columns:
            failed_ids = df['tess_id'].tolist()
        if 'original_row' in df.columns:
            failed_rows = df['original_row'].tolist()
            processed_rows.update(df['original_row'].tolist())
    
    return successful_ids, failed_ids, successful_rows, failed_rows, processed_rows

In [ ]:
def get_user_choice():
    while True:
        print("Do you wish to start from the beginning or append to existing data?")
        print("Options: 'start' to begin fresh, 'append' to continue from where you stopped")
        choice = input("Enter 'start' or 'append': ").strip().lower()
        if choice in ['start', 'append']:
            return choice
        print("Invalid input. Please enter 'start' or 'append'.")

In [ ]:
def get_start_row(total_rows):
    while True:
        try:
            start_from = input(f"Enter the original row number to start from (1 to {total_rows}, or press Enter for default): ").strip()
            if start_from == '':
                return 0
            start_from = int(start_from)
            if start_from < 1 or start_from > total_rows:
                print(f"Please enter a number between 1 and {total_rows}")
                continue
            return start_from - 1
        except ValueError:
            print("Invalid input. Please enter a valid number.")

In [ ]:
def get_how_many(total_remaining):
    while True:
        try:
            how_many = input(f"How many TIC IDs do you wish to process? (Remaining: {total_remaining}): ")
            how_many = int(how_many)
            if how_many <= 0:
                print("Please enter a positive number.")
                continue
            if how_many > total_remaining:
                print(f"Only {total_remaining} IDs remaining. Processing all remaining.")
                return total_remaining
            return how_many
        except ValueError:
            print("Invalid input. Please enter a valid number.")

In [ ]:
def continue_processing():
    while True:
        choice = input("Do you wish to process more files? (yes/y/no/n): ").strip().lower()
        if choice in ['yes', 'y']:
            return True
        elif choice in ['no', 'n']:
            return False
        print("Invalid input. Please enter 'yes', 'y', 'no', or 'n'.")

In [ ]:
def main():
    ctl = pd.read_csv("TESS_IDs.csv", header=0)
    all_tic_list = ctl.iloc[:, 0].astype(int).tolist()
    
    total_rows = len(all_tic_list)
    print(f"Total TIC IDs in file: {total_rows}")
    print(f"First 10 TIC IDs (with original row numbers):")
    for i in range(min(10, total_rows)):
        print(f"  Row {i+1}: TIC {all_tic_list[i]}")
    
    choice = get_user_choice()
    
    successful_ids = []
    failed_ids = []
    successful_rows = []
    failed_rows = []
    processed_rows = set()
    start_index = 0
    
    if choice == 'append':
        successful_ids, failed_ids, successful_rows, failed_rows, processed_rows = load_existing_data()
        print(f"Loaded existing data: {len(successful_ids)} successful, {len(failed_ids)} failed")
        
        if processed_rows:
            start_from = get_start_row(total_rows)
            start_index = start_from
            print(f"Starting from original row: {start_index + 1}")
        else:
            print("No existing data found. Starting from the beginning.")
            start_index = 0
    else:
        if os.path.exists('successful_tess_ids.csv'):
            os.remove('successful_tess_ids.csv')
        if os.path.exists('failed_tess_ids.csv'):
            os.remove('failed_tess_ids.csv')
        print("Starting fresh. All previous data will be overwritten.")
        start_index = 0
    
    if start_index > 0:
        remaining_indices = list(range(start_index, total_rows))
    else:
        remaining_indices = [i for i in range(total_rows) if (i + 1) not in processed_rows]
        start_index = min(remaining_indices) if remaining_indices else 0
    
    remaining_ids = [all_tic_list[i] for i in remaining_indices]
    total_remaining = len(remaining_ids)
    
    if total_remaining == 0:
        print("All IDs have already been processed!")
        print(f"Final results - Successful: {len(successful_ids)}, Failed: {len(failed_ids)}")
    else:
        print(f"Remaining rows to process: {total_remaining}")
        print(f"Starting from original row: {start_index + 1}")
        
        processing = True
        current_index = start_index
        
        while processing:
            how_many = get_how_many(total_remaining)
            
            batch_indices = list(range(current_index, min(current_index + how_many, total_rows)))
            batch_ids = [all_tic_list[i] for i in batch_indices]
            
            print("\n")
            print(f"Processing batch of {len(batch_ids)} IDs (Original rows {batch_indices[0] + 1} to {batch_indices[-1] + 1})")
            print("\n")
            
            for i, (row_idx, tic) in enumerate(zip(batch_indices, batch_ids)):
                original_row = row_idx + 1
                print(f"Processing {i+1}/{len(batch_ids)} - Original Row {original_row} - TIC {tic}")
                
                try:
                    clean_lc = process_star(tic)
                    if tic not in successful_ids and tic not in failed_ids:
                        successful_ids.append(tic)
                        successful_rows.append(original_row)
                    print(f"SUCCESS - Lightcurve valid")
                    
                except Exception as e:
                    if tic not in successful_ids and tic not in failed_ids:
                        failed_ids.append(tic)
                        failed_rows.append(original_row)
                    print(f"FAILED - {str(e)[:80]}")
                print()
            
            current_index += how_many
            total_remaining = total_rows - current_index
            
            save_ids(successful_rows, successful_ids, failed_rows, failed_ids)
            
            print("\n")
            print(f"Batch completed!")
            print(f"Processed rows: {current_index}/{total_rows}")
            print(f"Successful: {len(successful_ids)}")
            print(f"Failed: {len(failed_ids)}")
            print(f"Remaining rows: {total_remaining}")
            print("\n")
            
            if total_remaining == 0:
                print("All rows have been processed!")
                processing = False
            else:
                if not continue_processing():
                    processing = False
                    print("Stopping processing. Progress has been saved.")
    
    print("\n")
    print("VALIDATION COMPLETE")
    print("\n")
    print(f"Successful : {len(successful_ids)}")
    print(f"Failed     : {len(failed_ids)}")
    print(f"Success Rate : {len(successful_ids)/total_rows*100:.1f}%")
    print("\nFiles saved:")
    print("  - successful_tess_ids.csv (includes 'original_row' column)")
    if failed_ids:
        print("  - failed_tess_ids.csv (includes 'original_row' column)")
    
    print(f"\nFirst 10 successful TIC IDs (with original row numbers):")
    if len(successful_ids) >= 10:
        for i in range(10):
            print(f"  Row {successful_rows[i]}: TIC {successful_ids[i]}")
    else:
        for i, tic in enumerate(successful_ids):
            print(f"  Row {successful_rows[i]}: TIC {tic}")
    
    if failed_ids:
        print(f"\nFirst 10 failed TIC IDs (with original row numbers):")
        if len(failed_ids) >= 10:
            for i in range(10):
                print(f"  Row {failed_rows[i]}: TIC {failed_ids[i]}")
        else:
            for i, tic in enumerate(failed_ids):
                print(f"  Row {failed_rows[i]}: TIC {tic}")

In [ ]:
if __name__ == "__main__":
    main()